# Lesson 2 — Softmax, cross-entropy & multi-class classification

`demo.ipynb` trains a *binary* classifier with an SVM "max-margin" loss. Almost every modern classifier — including every
large language model predicting the next token — instead uses **softmax + cross-entropy**. In this lesson you build both
from the `exp` and `log` ops of Lesson 1, discover why the naive version crashes, and train a 3-class network.

**You will learn to**
1. Turn raw scores (*logits*) into probabilities with softmax
2. Make softmax numerically stable
3. Implement cross-entropy and discover its beautifully simple gradient
4. Sanity-check a loss at initialization, then train on a 3-class spiral

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import nanograd` works from the lessons/ folder without installing

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

In [ ]:
import math, random
import matplotlib.pyplot as plt
from nanograd.engine import Value
from nanograd.gradcheck import gradcheck

## Part 1 — From scores to probabilities

A network with `K` outputs produces `K` arbitrary real numbers called **logits**. To read them as probabilities we need them
to be positive and sum to 1. Softmax does exactly that:

$$\text{softmax}(l)_k = \frac{e^{l_k}}{\sum_j e^{l_j}}$$

`exp` makes everything positive (and preserves the order); dividing by the sum normalizes.

### Exercise 1.1 — naive softmax
Implement `softmax_naive(logits)` that takes a list of `Value`s and returns a list of `Value`s. Use `.exp()`.

In [ ]:
def softmax_naive(logits):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_softmax(fn):
    probs = fn([Value(1.0), Value(2.0), Value(3.0)])
    expected = [math.exp(k) / (math.exp(1) + math.exp(2) + math.exp(3)) for k in (1, 2, 3)]
    return all(abs(p.data - e) < 1e-12 for p, e in zip(probs, expected))

check("softmax_naive values", lambda: test_softmax(softmax_naive))
check("softmax_naive gradient", lambda: gradcheck(lambda v: softmax_naive(v)[0] * 2 + softmax_naive(v)[2], [0.5, -1.0, 2.0]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def softmax_naive(logits):
    exps = [l.exp() for l in logits]
    total = sum(exps)
    return [e / total for e in exps]

check("softmax_naive values", lambda: test_softmax(softmax_naive))
check("softmax_naive gradient", lambda: gradcheck(lambda v: softmax_naive(v)[0] * 2 + softmax_naive(v)[2], [0.5, -1.0, 2.0]))

## Part 2 — When softmax explodes

Logits are unbounded. A network that is very confident can easily produce a logit of a few hundred, and $e^{710}$ is already
larger than the biggest float64:

In [ ]:
try:
    softmax_naive([Value(1000.0), Value(1001.0)])
except OverflowError as e:
    print("OverflowError:", e)

The fix is a small piece of algebra. For any constant $m$:

$$\frac{e^{l_k}}{\sum_j e^{l_j}} = \frac{e^{-m}\,e^{l_k}}{e^{-m}\sum_j e^{l_j}} = \frac{e^{l_k - m}}{\sum_j e^{l_j - m}}$$

Choose $m = \max_j l_j$. Now the largest exponent is $e^0 = 1$ and nothing can overflow.

### Exercise 2.1 — stable softmax
Implement `softmax(logits)` using the max trick. *Hint: `m` should be a plain float (`max(l.data for l in logits)`) — a constant
shift needs no gradient, and as the algebra shows, the output does not depend on it.*

In [ ]:
def softmax(logits):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("softmax values", lambda: test_softmax(softmax))
check("softmax huge logits", lambda: abs(softmax([Value(1000.0), Value(1001.0)])[1].data - 1 / (1 + math.exp(-1))) < 1e-12)
check("softmax gradient", lambda: gradcheck(lambda v: softmax(v)[0] * 2 + softmax(v)[2], [0.5, -1.0, 2.0]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def softmax(logits):
    m = max(l.data for l in logits)
    exps = [(l - m).exp() for l in logits]
    total = sum(exps)
    return [e / total for e in exps]

check("softmax values", lambda: test_softmax(softmax))
check("softmax huge logits", lambda: abs(softmax([Value(1000.0), Value(1001.0)])[1].data - 1 / (1 + math.exp(-1))) < 1e-12)
check("softmax gradient", lambda: gradcheck(lambda v: softmax(v)[0] * 2 + softmax(v)[2], [0.5, -1.0, 2.0]))

## Part 3 — Cross-entropy

If the correct class is `t`, the loss is the **negative log probability** the model gave it:

$$L = -\log p_t$$

* Perfectly confident and right: $p_t = 1 \Rightarrow L = 0$.
* Confidently wrong: $p_t \to 0 \Rightarrow L \to \infty$. Cross-entropy punishes confident mistakes *very* hard.

In [ ]:
ps = [i / 100 for i in range(1, 100)]
plt.figure(figsize=(5, 3))
plt.plot(ps, [-math.log(p) for p in ps])
plt.xlabel("probability given to the correct class, $p_t$"); plt.ylabel("loss  $-\\log p_t$")
plt.title("cross-entropy for a single example"); plt.grid(alpha=0.3)
plt.show()

### Exercise 3.1 — cross-entropy
Implement `cross_entropy(logits, target)` where `target` is an `int`.

Taking `log(softmax(...))` works, but it's wasteful and can produce `log(0)` when a probability underflows to zero. Instead,
simplify the log of the stable softmax by hand:

$$\log p_t = (l_t - m) - \log \sum_j e^{l_j - m}$$

The second term is called **log-sum-exp**.

In [ ]:
def cross_entropy(logits, target):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_ce_value():
    logits = [Value(2.0), Value(0.5), Value(-1.0)]
    expected = -math.log(math.exp(2.0) / (math.exp(2.0) + math.exp(0.5) + math.exp(-1.0)))
    return abs(cross_entropy(logits, 0).data - expected) < 1e-12

check("cross_entropy value", test_ce_value)
check("cross_entropy gradient", lambda: all(gradcheck(lambda v: cross_entropy(v, t), [0.3, -1.1, 2.0]) for t in range(3)))
check("cross_entropy huge logits", lambda: abs(cross_entropy([Value(1000.0), Value(0.0)], 0).data) < 1e-12)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def cross_entropy(logits, target):
    m = max(l.data for l in logits)
    shifted = [l - m for l in logits]
    logsumexp = sum(s.exp() for s in shifted).log()
    return -(shifted[target] - logsumexp)

check("cross_entropy value", test_ce_value)
check("cross_entropy gradient", lambda: all(gradcheck(lambda v: cross_entropy(v, t), [0.3, -1.1, 2.0]) for t in range(3)))
check("cross_entropy huge logits", lambda: abs(cross_entropy([Value(1000.0), Value(0.0)], 0).data) < 1e-12)

### Exercise 3.2 — the gradient of cross-entropy

Here is a remarkable fact worth discovering yourself. Run backprop through your `cross_entropy` and print the gradient of each
logit next to the softmax probabilities. Do you see the pattern?

In [ ]:
xs, t = [2.0, 0.5, -1.0], 1
logits = [Value(x) for x in xs]
cross_entropy(logits, t).backward()
probs = [p.data for p in softmax([Value(x) for x in xs])]
for k in range(3):
    print(f"class {k}: prob {probs[k]:.4f}   dL/dlogit {logits[k].grad:+.4f}   {'<- target' if k == t else ''}")

Now write `expected_grad(xs, target)` that returns the gradient using a **closed-form formula** in terms of the probabilities —
no autograd allowed. (Plain floats in, list of floats out.)

In [ ]:
def expected_grad(xs, target):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_expected_grad():
    xs, t = [0.3, -1.1, 2.0, 0.7], 2
    logits = [Value(x) for x in xs]
    cross_entropy(logits, t).backward()
    return all(abs(l.grad - g) < 1e-12 for l, g in zip(logits, expected_grad(xs, t)))

check("expected_grad", test_expected_grad)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def expected_grad(xs, target):
    # dL/dl_k = p_k - [k == target]   ("prediction minus truth")
    probs = [p.data for p in softmax([Value(x) for x in xs])]
    return [p - (1 if k == target else 0) for k, p in enumerate(probs)]

check("expected_grad", test_expected_grad)

**The gradient is simply *prediction minus truth*:** $\frac{\partial L}{\partial l_k} = p_k - \mathbb{1}[k = t]$.
Every wrong class is pushed down in proportion to how much probability it stole; the right class is pulled up by however much
it was missing. This simple, well-behaved gradient (it never exceeds 1 in magnitude) is a big part of why cross-entropy trains so
well. It is also why frameworks fuse softmax + cross-entropy into one op: the combined backward pass is one subtraction.

From now on we use the library versions in `nanograd.losses` — they are identical to yours.

In [ ]:
from nanograd.losses import softmax, cross_entropy

## Part 4 — Training a 3-class classifier

Let's classify the classic *spiral* dataset: three interleaved arms that no straight line can separate.

In [ ]:
from nanograd.nn import MLP
from nanograd.datasets import make_spiral
from nanograd.viz import plot_decision_boundary

X, y = make_spiral(n_per_class=30, n_classes=3, noise=0.2, seed=0)
plt.figure(figsize=(4, 4))
plt.scatter([p[0] for p in X], [p[1] for p in X], c=y, cmap=plt.cm.Spectral, s=20, edgecolors='k', linewidths=0.3)
plt.title("spiral: 3 classes"); plt.show()

In [ ]:
random.seed(1337)
model = MLP(2, [16, 16, 3])   # 3 outputs: one logit per class
print(model)
print("number of parameters", len(model.parameters()))

def loss_and_accuracy(model, X, y):
    logits = [model([Value(xi) for xi in x]) for x in X]
    loss = sum(cross_entropy(l, t) for l, t in zip(logits, y)) * (1.0 / len(X))
    preds = [max(range(3), key=lambda k: l[k].data) for l in logits]
    acc = sum(p == t for p, t in zip(preds, y)) / len(y)
    return loss, acc

### Exercise 4.1 — what should the loss be *before* training?

A freshly initialized network knows nothing, so (roughly) it should give each of the 3 classes equal probability.
Before running the next cell, work out what cross-entropy that implies and write it as a formula.
This "loss at init" check catches a surprising number of bugs (wrong labels, wrong loss, bad initialization) before you waste
time training.

In [ ]:
expected_initial_loss = None  # YOUR ANSWER HERE (a formula, e.g. using math.log)

In [ ]:
loss, acc = loss_and_accuracy(model, X, y)
print(f"actual initial loss: {loss.data:.4f}   accuracy {acc:.0%}")

def test_initial_loss():
    if expected_initial_loss is None:
        raise NotImplementedError
    print(f"   your expected loss: {expected_initial_loss:.4f}")
    return abs(expected_initial_loss - math.log(3)) < 1e-6

check("expected initial loss", test_initial_loss)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
expected_initial_loss = -math.log(1 / 3)   # = log(3) ~ 1.0986: each class gets probability 1/3
check("expected initial loss", test_initial_loss)

Our actual initial loss is a bit higher than $\ln 3$, because the random weights make the logits not quite equal — the network
starts out slightly confident, and sometimes confidently wrong. (Lesson 4 looks at how initialization affects this.)

Now train with plain gradient descent. This takes about 20 seconds: every one of the ~65,000 scalar operations is a Python object!

In [ ]:
losses = []
for k in range(60):
    loss, acc = loss_and_accuracy(model, X, y)
    model.zero_grad()
    loss.backward()
    for p in model.parameters():
        p.data -= 0.3 * p.grad
    losses.append(loss.data)
    if k % 10 == 0 or k == 59:
        print(f"step {k:2d}  loss {loss.data:.4f}  accuracy {acc:.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(losses); axes[0].axhline(math.log(3), ls='--', c='gray', label='log(3): uniform guess')
axes[0].set_xlabel('step'); axes[0].set_ylabel('cross-entropy'); axes[0].legend(); axes[0].set_title('training loss')
plot_decision_boundary(model, X, y, ax=axes[1], h=0.06, title='decision regions')
plt.show()

### Exercise 4.2 — confidence on a test point (open-ended)
Pick a point in the middle of an arm and one in a gap between arms. Print `softmax(model(point))` for each.
Where is the model confident? Where is it unsure? What would happen to the loss if the model were confidently wrong on a training point?

In [ ]:
# YOUR CODE HERE

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
for point in ([0.0, 0.9], [0.05, 0.05], [-0.6, -0.2]):
    probs = softmax(model([Value(p) for p in point]))
    print(point, "->", " ".join(f"{p.data:.2f}" for p in probs))
# Points deep inside an arm get one probability near 1. Points near the centre, where the arms
# meet, get a spread-out distribution. A confidently wrong training point (p_t ~ 0.01) would
# contribute -log(0.01) ~ 4.6 to the loss, ~40x more than a confident correct one.

## Summary
* **Softmax** turns logits into probabilities; subtract the max logit so `exp` cannot overflow.
* **Cross-entropy** $=-\log p_t$, computed stably with log-sum-exp.
* Its gradient is **prediction − truth**, which is why the pair trains so well and is fused in real frameworks.
* Check the **loss at init** ($\ln K$ for $K$ classes) before you train.
* A language model is this exact setup with $K$ = the vocabulary size — see Karpathy's *microgpt* for the next step.

Next lesson: we replace the hand-written `p.data -= lr * p.grad` with real optimizers.